# SECTION 00 · Cài đặt & Kết nối Hệ thống

- **Người phụ trách:** Trần Quang Minh
- **Đầu vào:** Thông số cấu hình SQL Server (Server: localhost hoặc tên máy, Database: malaysia_inflation)
- **Đầu ra:** Biến ROOT, các đường dẫn (RAW_DIR, PROC_DIR), đối tượng engine kết nối SQL Server và hàm tiện ích sql()
- **Liên quan proposal:** Mục 2 — Thiết lập môi trường & hạ tầng CSDL

In [1]:
import requests
import pandas as pd
import pathlib
import sys
import sqlalchemy as sa
import pyodbc
import warnings

warnings.filterwarnings('ignore')
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / 'requirements.txt').exists()),
    pathlib.Path.cwd()
)
RAW_DIR = ROOT / 'data' / 'raw'
PROC_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'output'

for folder in (RAW_DIR, PROC_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SERVER = r'localhost'  # SQL Server mặc định trên chính máy đang chạy code
DATABASE = 'malaysia_inflation'

DRIVER = next(
    (d for d in ('ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server')
     if d in pyodbc.drivers()),
    None
)
if DRIVER is None:
    raise RuntimeError('Cần cài ODBC Driver 17 hoặc 18 for SQL Server.')

def odbc_string(database):
    return (
        f'DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={database};'
        'Trusted_Connection=yes;TrustServerCertificate=yes;'
    )

with pyodbc.connect(odbc_string('master'), autocommit=True) as conn:
    conn.execute(f"IF DB_ID(N'{DATABASE}') IS NULL CREATE DATABASE [{DATABASE}];")

connection_url = sa.engine.URL.create(
    'mssql+pyodbc', query={'odbc_connect': odbc_string(DATABASE)}
)
engine = sa.create_engine(connection_url, fast_executemany=True, pool_pre_ping=True)

def sql(query):
    with engine.connect() as conn:
        return pd.read_sql_query(sa.text(query), conn)

def sql_dtypes(dataframe):
    return {
        col: sa.types.NVARCHAR(length=255)
        for col in dataframe.select_dtypes(include=['object', 'string']).columns
    }

print(sql('SELECT @@SERVERNAME AS server_name, DB_NAME() AS database_name;'))
print(f'ROOT: {ROOT}')

       server_name       database_name
0  LAPTOP-AVBTJEN9  malaysia_inflation
ROOT: C:\Users\ASUS\OneDrive\Tài liệu\project ADY\.codex-final-test\ThanhVien1_DataEngineer


# SECTION 01 · Thu thập dữ liệu từ API (Data Ingestion)

- **Người phụ trách:** Trần Quang Minh
- **Đầu vào:** API OpenDOSM (dataset: cpi_headline, cpi_state, population_state)
- **Đầu ra:** 3 file CSV thô lưu tại data/raw/ (cpi_headline.csv, cpi_category.csv, population_state.csv)
- **Liên quan proposal:** Mục 3 — Nguồn dữ liệu & Thu thập tự động

In [2]:
API_URL = 'https://api.data.gov.my/data-catalogue'

def goi_api(dataset, **filters):
    params = {'id': dataset, 'limit': 100_000, **filters}
    response = requests.get(API_URL, params=params, timeout=120)
    response.raise_for_status()
    payload = response.json()
    records = payload.get('data', payload) if isinstance(payload, dict) else payload
    return pd.DataFrame(records)

COICOP_MAP = {
    '01': 'Thực phẩm và đồ uống',
    '02': 'Rượu bia và thuốc lá',
    '03': 'Quần áo và giày dép',
    '04': 'Nhà ở, nước, điện, gas và nhiên liệu',
    '05': 'Đồ dùng và thiết bị gia đình',
    '06': 'Y tế',
    '07': 'Giao thông',
    '08': 'Thông tin và truyền thông',
    '09': 'Giải trí, thể thao và văn hóa',
    '10': 'Giáo dục',
    '11': 'Nhà hàng và dịch vụ lưu trú',
    '12': 'Bảo hiểm và dịch vụ tài chính',
    '13': 'Chăm sóc cá nhân và hàng hóa khác',
}

cpi_national = goi_api('cpi_headline')
cpi_my = cpi_national.loc[cpi_national['division'].eq('overall')].assign(state='Malaysia')
cpi_states = goi_api('cpi_state', filter='overall@division')

df_cpi_raw = (
    pd.concat([cpi_my, cpi_states], ignore_index=True)
    [['date', 'state', 'division', 'index']]
    .rename(columns={'index': 'index_val'})
)

df_cat_raw = cpi_national.loc[cpi_national['division'].ne('overall')].copy()
df_cat_raw['category'] = df_cat_raw['division'].map(COICOP_MAP)
df_cat_raw = (
    df_cat_raw[['date', 'division', 'category', 'index']]
    .rename(columns={'index': 'index_val'})
)

df_pop_raw = goi_api('population_state')

df_cpi_raw.to_csv(RAW_DIR / 'cpi_headline.csv', index=False, encoding='utf-8-sig')
df_cat_raw.to_csv(RAW_DIR / 'cpi_category.csv', index=False, encoding='utf-8-sig')
df_pop_raw.to_csv(RAW_DIR / 'population_state.csv', index=False, encoding='utf-8-sig')

print(f'cpi_headline.csv    : {len(df_cpi_raw):,} dòng')
print(f'cpi_category.csv    : {len(df_cat_raw):,} dòng')
print(f'population_state.csv: {len(df_pop_raw):,} dòng')

cpi_headline.csv    : 3,760 dòng
cpi_category.csv    : 7,280 dòng
population_state.csv: 25,536 dòng


# SECTION 02 · Lưu trữ thô vào SQL Server (Raw Storage)

- **Người phụ trách:** Trần Quang Minh
- **Đầu vào:** 3 DataFrame thô từ Section 01
- **Đầu ra:** 3 bảng raw trong CSDL SQL Server (raw_cpi_headline, raw_cpi_category, raw_population)
- **Liên quan proposal:** Mục 4 — Lưu trữ thô & Quản trị CSDL

In [3]:
raw_tables = {
    'raw_cpi_headline': df_cpi_raw,
    'raw_cpi_category': df_cat_raw,
    'raw_population': df_pop_raw,
}

for table_name, dataframe in raw_tables.items():
    dataframe.to_sql(
        table_name, engine, schema='dbo',
        if_exists='replace', index=False, chunksize=1_000,
        dtype=sql_dtypes(dataframe)
    )

sql("""
SELECT 'raw_cpi_headline' AS table_name, COUNT(*) AS row_count
FROM dbo.raw_cpi_headline
UNION ALL
SELECT 'raw_cpi_category', COUNT(*) FROM dbo.raw_cpi_category
UNION ALL
SELECT 'raw_population', COUNT(*) FROM dbo.raw_population;
""")

,table_name,row_count
0,raw_cpi_headline,3760
1,raw_cpi_category,7280
2,raw_population,25536


# SECTION 03 · Làm sạch dữ liệu (Data Cleaning)

- **Người phụ trách:** Trần Quang Minh
- **Đầu vào:** 3 bảng raw_cpi_headline, raw_cpi_category và raw_population từ SQL Server
- **Đầu ra:** 3 bảng sạch và 3 file CSV trong data/processed/ để bàn giao cho Thành viên 2 và 3
- **Liên quan proposal:** Mục 5 — Tiền xử lý & Chuẩn hóa dữ liệu

In [4]:
cpi = sql('SELECT * FROM dbo.raw_cpi_headline')
category = sql('SELECT * FROM dbo.raw_cpi_category')
population = sql('SELECT * FROM dbo.raw_population')
rows_before = [len(cpi), len(category), len(population)]

# CPI tổng
cpi['date'] = pd.to_datetime(cpi['date'], errors='coerce')
cpi['index_val'] = pd.to_numeric(cpi['index_val'], errors='coerce')
cpi['state'] = (cpi['state'].astype('string').str.strip()
                .replace('', pd.NA).fillna('Malaysia')
                .str.replace(r'^W\.P\.\s*', '', regex=True))
cpi = (cpi[cpi['index_val'].between(50, 300)].dropna().drop_duplicates()
       .sort_values(['state', 'date']).reset_index(drop=True))

# CPI nhóm hàng
category['date'] = pd.to_datetime(category['date'], errors='coerce')
category['index_val'] = pd.to_numeric(category['index_val'], errors='coerce')
for col in ['division', 'category']:
    category[col] = category[col].astype('string').str.strip()
category = (category[category['index_val'].gt(0) & category['index_val'].le(300)]
            .dropna().drop_duplicates()
            .sort_values(['division', 'date']).reset_index(drop=True))

# Dân số
population['date'] = pd.to_datetime(population['date'], errors='coerce')
population['population'] = pd.to_numeric(population['population'], errors='coerce')
for col in ['state', 'age', 'sex', 'ethnicity']:
    population[col] = population[col].astype('string').str.strip()
population['state'] = population['state'].str.replace(r'^W\.P\.\s*', '', regex=True)
population = (population[population['population'].ge(0)].dropna().drop_duplicates()
              .sort_values(['state', 'date', 'sex', 'age', 'ethnicity'])
              .reset_index(drop=True))

clean_data = {
    'cpi_clean': (cpi, 'cpi_clean.csv'),
    'cpi_category_clean': (category, 'cpi_category_clean.csv'),
    'population_clean': (population, 'population_clean.csv'),
}
for table, (df, filename) in clean_data.items():
    df.to_sql(
        table, engine, schema='dbo', if_exists='replace', index=False,
        dtype=sql_dtypes(df)
    )
    df.to_csv(PROC_DIR / filename, index=False, encoding='utf-8-sig')

# Bản Excel để cột age như 10-14 không bị tự đổi thành ngày 14-Oct.
population.to_excel(
    PROC_DIR / 'population_clean_excel.xlsx', index=False, freeze_panes=(1, 0)
)

for (table, (df, _)), before in zip(clean_data.items(), rows_before):
    print(f'{table:<22}: {before:>6,} → {len(df):>6,} dòng')
cpi.head(3)

cpi_clean             :  3,760 →  3,714 dòng
cpi_category_clean    :  7,280 →  5,816 dòng
population_clean      : 25,536 → 25,536 dòng


,date,state,division,index_val
0,2010-01-01,Johor,overall,99.4
1,2010-02-01,Johor,overall,99.4
2,2010-03-01,Johor,overall,99.4
